In [61]:
# Standard libraries
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Set, Dict
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Reproducibility
#RANDOM_SEED = 42
#np.random.seed(RANDOM_SEED)

# RDKitran
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs

# Scikit-learn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

print("Core imports successful!")
print(f"RDKit version: {Chem.rdBase.rdkitVersion}")


# ChemProp import (may take a few seconds)
try:
    import chemprop
    print(f"ChemProp version: {chemprop.__version__}")
    CHEMPROP_AVAILABLE = True
except ImportError:
    print("ChemProp not installed. Install with: pip install chemprop")
    CHEMPROP_AVAILABLE = False

Core imports successful!
RDKit version: 2023.09.6
ChemProp not installed. Install with: pip install chemprop


In [68]:
# RF + Morgan fingerprintings

from random import random


def compute_morgan_fingerprint(mol: Chem.Mol, radius: int = 2, n_bits: int = 2048) -> np.ndarray:
    """Compute Morgan (ECFP) fingerprint.
    
    Args:
        mol: RDKit Mol object
        radius: Fingerprint radius (2 == ECFP4)
        n_bits: Number of bits (aka the 'length' of the fingerprint)
        
    Returns:
        numpy array of shape (n_bits,)
    """
    fingerpint = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    fingerpint = np.array(fingerpint)
    return fingerpint

def train_random_forest(
    X_train,y_train,X_val,y_val,X_test,y_test,
    n_estimators: int = 100,
    random_state: int = None ) -> Dict:
    if random_state is None:
        random_state = np.random.randint(0, 10000)  # Actually call the function
    
    print("random state :" + str(random_state))
    model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state) #defining the model with the wanted hyperparameters
    model.fit(X_train, y_train) #fitting the model to the hyperparameters and the training data
    
    # Predictions
    pred_train = model.predict(X_train)
    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, pred_train))
    val_rmse = np.sqrt(mean_squared_error(y_val, pred_val))
    test_rmse = np.sqrt(mean_squared_error(y_test, pred_test))
    test_r2 = r2_score(y_test, pred_test)
    test_rho, _ = spearmanr(y_test, pred_test)
    test_mae = mean_absolute_error(y_test, pred_test)
    
    return {
        'model': model,
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'test_rmse': test_rmse,
        'test_r2': test_r2,
        'test_rho': test_rho,
        'test_predictions': pred_test,
        'test_measured': y_test,
        'test_mae': test_mae
    }

Now same process on the other datasets :

In [69]:
def run_randomforest (dataset,target,state): 
    test = pd.read_csv(f'../data/{dataset}_splits/{dataset}_test.csv')
    train = pd.read_csv(f'../data/{dataset}_splits/{dataset}_train.csv')
    val = pd.read_csv(f'../data/{dataset}_splits/{dataset}_val.csv')

    test['mol2'] = test['smiles'].apply(Chem.MolFromSmiles)
    train['mol2'] = train['smiles'].apply(Chem.MolFromSmiles)
    val['mol2'] = val['smiles'].apply(Chem.MolFromSmiles)

    test['mol'] = test['mol2']
    train['mol'] = train['mol2']
    val['mol'] = val['mol2']

    X_train = np.array([compute_morgan_fingerprint(mol) for mol in train['mol']])
    y_train = train[target].values
    X_val = np.array([compute_morgan_fingerprint(mol) for mol in val['mol']])
    y_val = val[target].values
    X_test = np.array([compute_morgan_fingerprint(mol) for mol in test['mol']])
    y_test = test[target].values  


    rf_morgan_scaffold = train_random_forest(X_train,y_train,X_val,y_val,X_test,y_test,random_state=state)
    print(f"  Test RMSE: {rf_morgan_scaffold['test_rmse']:.3f}, R²: {rf_morgan_scaffold['test_r2']:.3f}, ρ: {rf_morgan_scaffold['test_rho']:.3f}, MAE: {rf_morgan_scaffold['test_mae']:.3f}")
    return rf_morgan_scaffold 


In [72]:
results = {}

for dataset_duo in [['lipo', 'exp'], ['qm7', 'u0_atom'], ['dr', 'DR']]:
    dataset = dataset_duo[0]
    target = dataset_duo[1] 
    results[dataset] = {'rmse': [], 'r2': [], 'rho': [], 'mae': []}
    print(f"Running Random Forest on {dataset} with target {target}...")
    for state in range (5) :
        result = run_randomforest(dataset,target,state)
        results[dataset]['rmse'].append(result['test_rmse'])
        results[dataset]['r2'].append(result['test_r2'])
        results[dataset]['rho'].append(result['test_rho'])
        results[dataset]['mae'].append(result['test_mae'])
    print(f"\n{dataset.upper()} Summary (5 runs):")
    for metric_name, metric_key in [('RMSE', 'rmse'), ('R²', 'r2'), ('ρ', 'rho'), ('MAE', 'mae')]:
        mean = np.mean(results[dataset][metric_key])
        std = np.std(results[dataset][metric_key], ddof=1)
        print(f"  {metric_name}: {mean:.3f} ± {std:.3f}")    

Running Random Forest on lipo with target exp...
random state :0
  Test RMSE: 0.871, R²: 0.430, ρ: 0.644, MAE: 0.683
random state :1
  Test RMSE: 0.869, R²: 0.432, ρ: 0.649, MAE: 0.680
random state :2
  Test RMSE: 0.872, R²: 0.429, ρ: 0.643, MAE: 0.678
random state :3
  Test RMSE: 0.876, R²: 0.423, ρ: 0.649, MAE: 0.679
random state :4
  Test RMSE: 0.873, R²: 0.427, ρ: 0.649, MAE: 0.684

LIPO Summary (5 runs):
  RMSE: 0.872 ± 0.003
  R²: 0.428 ± 0.003
  ρ: 0.647 ± 0.003
  MAE: 0.681 ± 0.002
Running Random Forest on qm7 with target u0_atom...


[15:04:56] WARNING: not removing hydrogen atom without neighbors
[15:04:56] WARNING: not removing hydrogen atom without neighbors


random state :0
  Test RMSE: 193.630, R²: 0.229, ρ: 0.553, MAE: 141.011


[15:06:24] WARNING: not removing hydrogen atom without neighbors
[15:06:24] WARNING: not removing hydrogen atom without neighbors


random state :1
  Test RMSE: 192.517, R²: 0.238, ρ: 0.560, MAE: 140.209


...

random state :2
  Test RMSE: 194.182, R²: 0.225, ρ: 0.551, MAE: 141.820


[15:09:23] WARNING: not removing hydrogen atom without neighbors
[15:09:23] WARNING: not removing hydrogen atom without neighbors


random state :3
  Test RMSE: 194.711, R²: 0.220, ρ: 0.549, MAE: 141.666


[15:11:08] WARNING: not removing hydrogen atom without neighbors
[15:11:08] WARNING: not removing hydrogen atom without neighbors


random state :4
  Test RMSE: 193.889, R²: 0.227, ρ: 0.553, MAE: 141.268

QM7 Summary (5 runs):
  RMSE: 193.786 ± 0.815
  R²: 0.228 ± 0.006
  ρ: 0.553 ± 0.004
  MAE: 141.195 ± 0.637
Running Random Forest on dr with target DR...
random state :0
  Test RMSE: 0.471, R²: 0.235, ρ: 0.487, MAE: 0.364
random state :1
  Test RMSE: 0.461, R²: 0.268, ρ: 0.479, MAE: 0.353
random state :2
  Test RMSE: 0.464, R²: 0.256, ρ: 0.480, MAE: 0.359
random state :3
  Test RMSE: 0.473, R²: 0.228, ρ: 0.474, MAE: 0.363
random state :4
  Test RMSE: 0.465, R²: 0.254, ρ: 0.488, MAE: 0.361

DR Summary (5 runs):
  RMSE: 0.467 ± 0.005
  R²: 0.248 ± 0.016
  ρ: 0.482 ± 0.006
  MAE: 0.360 ± 0.004
